[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/5_SAM.ipynb)

**Día 2 · Cuaderno 2 de 3** · Acompaña al tema 5 de la web: [Segmentación con SAM](https://datamecum.sergiovillanuevalopez.com/temas/05-sam/)

**En clase:** hasta la búsqueda semántica. **Extra**, si da tiempo: el perímetro del lago de Central Park.

# SAM 2: segmentar cualquier objeto

**SAM 2** (*Segment Anything Model 2*, Meta, 2024) recorta cualquier objeto de una imagen a nivel de píxel. No sabe qué es el objeto: le indicas dónde está con un punto o una caja, y devuelve su máscara exacta.

- Funciona con objetos que no ha visto nunca.
- Admite puntos positivos ("esto es el objeto") y negativos ("esto no").
- Combinado con un detector por texto, segmenta lo que le pidas con palabras.

**Antes de empezar**

1. Activa la GPU: menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.
2. Ejecuta las celdas en orden, de arriba abajo, con `Mayúsculas + Intro`.
3. No hay que instalar nada: todas las librerías que usamos ya vienen en Colab. Los modelos se descargan solos desde Hugging Face la primera vez que se usan.

Este cuaderno es tuyo: al abrirlo desde el enlace se crea una copia que solo ves tú, y puedes cambiar lo que quieras sin estropear el original.

## Preparación

In [ ]:
import json
import torch
import requests
import transformers
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from pathlib import Path
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Usando:", device)
transformers.logging.set_verbosity_error()   # oculta avisos internos que no afectan al resultado

ASSETS = "https://raw.githubusercontent.com/sergiovillanueva/Modelos_Fundacionales/main/assets/"


def cargar_imagen(nombre_o_url):
    """Abre una imagen del curso en RGB.

    Si el cuaderno se ejecuta dentro de la carpeta del curso descargada (por ejemplo, sin internet),
    la toma de assets/; si no, la descarga del repositorio del curso o de la URL indicada.
    """
    local = Path("assets") / Path(nombre_o_url).name
    if local.exists():
        return Image.open(local).convert("RGB")
    url = nombre_o_url if nombre_o_url.startswith("http") else ASSETS + nombre_o_url
    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()
    return Image.open(BytesIO(respuesta.content)).convert("RGB")


image_person_dog = cargar_imagen("person_dog.jpg")
image_fruits = cargar_imagen("fruits.jpg")
image_fruits3 = cargar_imagen("fruits3.jpg")
image_person_cars = cargar_imagen("person_cars.jpg")
image_cars = cargar_imagen("cars.jpg")
image_board = cargar_imagen("board.jpg")
image_park = cargar_imagen("park.jpg")
print("Imágenes cargadas")

## Cargar SAM 2

Hay cuatro tamaños. Usamos `small`: va rápido y recorta bien.

| Tamaño | Descarga |
|---|---|
| tiny | 156 MB |
| small | 184 MB |
| base-plus | 323 MB |
| large | 898 MB, el mejor contorno |

In [ ]:
from transformers import Sam2Model, Sam2Processor

sam_name = "facebook/sam2.1-hiera-small"
sam_processor = Sam2Processor.from_pretrained(sam_name)
sam_model = Sam2Model.from_pretrained(sam_name).to(device)
print("SAM 2 cargado:", sam_name)

## Segmentar con un punto

Le damos un punto `(x, y)` en píxeles, contando desde la esquina superior izquierda, y una etiqueta: `1` si el punto está sobre el objeto y `0` si está fuera.

SAM devuelve **tres máscaras** para cada indicación, de más pequeña a más grande (por ejemplo: un ojo, la cara, la persona entera), y una puntuación de calidad para cada una. Nos quedamos con la mejor.

Las listas van muy anidadas porque SAM admite varias imágenes, varios objetos por imagen y varios puntos por objeto. `segmentar` lo esconde para que solo haya que pasar los puntos, y `mostrar_mascara` enseña el resultado; la usaremos en todo el cuaderno.

In [ ]:
def mostrar_mascara(imagen, mascara, titulo, puntos=(), etiquetas=()):
    """Muestra la imagen con los puntos, el objeto y todo lo demás."""
    figura, ejes = plt.subplots(1, 3, figsize=(17, 5))
    ejes[0].imshow(imagen)
    ejes[0].set_title("Original")
    for (x, y), etiqueta in zip(puntos, etiquetas):   # verde: objeto, rojo: fuera
        ejes[0].scatter(x, y, s=250, marker="*", c="lime" if etiqueta == 1 else "red", edgecolors="black")
    ejes[1].imshow(np.array(imagen) * mascara[:, :, None])
    ejes[1].set_title(titulo)
    ejes[2].imshow(np.array(imagen) * ~mascara[:, :, None])
    ejes[2].set_title("Todo lo demás")
    for eje in ejes:
        eje.axis("off")
    plt.show()


def segmentar(imagen, puntos, etiquetas, mostrar=True):
    """Segmenta con SAM 2 a partir de puntos [[x, y], ...] y etiquetas [1, 0, ...]."""
    entradas = sam_processor(images=imagen, input_points=[[puntos]], input_labels=[[etiquetas]],
                             return_tensors="pt").to(device)
    with torch.no_grad():
        salidas = sam_model(**entradas)
    mascaras = sam_processor.post_process_masks(salidas.pred_masks.cpu(), entradas["original_sizes"])[0][0]
    mejor = salidas.iou_scores[0, 0].argmax().item()   # de las 3 máscaras, la de mejor calidad
    mascara = mascaras[mejor].numpy()
    if mostrar:
        mostrar_mascara(imagen, mascara, f"Máscara {mejor + 1} de 3", puntos, etiquetas)
    return mascara


mascara_perro = segmentar(image_person_dog, [[287, 537]], [1])

## Afinar con varios puntos

Un solo punto es ambiguo. Si pinchamos en el vaquero de la mujer, SAM entiende que queremos el pantalón. Añadiendo un segundo punto positivo en el jersey le decimos que queremos a la persona entera. Y con un punto negativo en la cara le pedimos que la quite.

In [ ]:
_ = segmentar(image_person_dog, [[520, 560]], [1])                                # el pantalón
_ = segmentar(image_person_dog, [[520, 560], [460, 360]], [1, 1])                 # la persona entera
_ = segmentar(image_person_dog, [[520, 560], [460, 360], [420, 280]], [1, 1, 0])  # la persona sin la cabeza

### Buscar el punto con deslizadores

Mueve el punto por la imagen y mira cómo cambia la máscara. Cada posición es una segmentación nueva, calculada en el momento: fíjate en cuándo SAM salta a otro objeto y cuándo amplía el que ya tenía.

In [ ]:
from ipywidgets import IntSlider, interact

ancho, alto = image_fruits3.size

@interact(x=IntSlider(min=0, max=ancho - 1, step=20, value=500),
          y=IntSlider(min=0, max=alto - 1, step=20, value=400))
def mover_punto(x, y):
    mascara = segmentar(image_fruits3, [[x, y]], [1], mostrar=False)
    mostrar_mascara(image_fruits3, mascara, f"El {100 * mascara.mean():.1f} % de la imagen", [[x, y]], [1])

## Segmentar por texto: Grounding DINO + SAM 2

SAM no entiende palabras, pero podemos encadenarlo con un modelo que sí: **Grounding DINO**, el detector por texto del cuaderno 3.

1. Grounding DINO encuentra los objetos que nombras y devuelve sus cajas.
2. SAM recibe esas cajas y devuelve la máscara de cada una.

Recuerda el formato del texto: en inglés, en minúsculas y cada objeto terminado en punto.

In [ ]:
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

dino_name = "IDEA-Research/grounding-dino-base"
dino_processor = AutoProcessor.from_pretrained(dino_name)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_name).to(device)
print("Grounding DINO cargado")

In [ ]:
def mascaras_de_cajas(imagen, cajas):
    """SAM 2 a partir de cajas [x1, y1, x2, y2]: devuelve la unión de las máscaras."""
    union = np.zeros((imagen.height, imagen.width), dtype=bool)
    if not cajas:
        return union
    entradas = sam_processor(images=imagen, input_boxes=[cajas], return_tensors="pt").to(device)
    with torch.no_grad():
        salidas = sam_model(**entradas)
    mascaras = sam_processor.post_process_masks(salidas.pred_masks.cpu(), entradas["original_sizes"])[0]
    for tres_mascaras, puntuaciones in zip(mascaras, salidas.iou_scores[0].cpu()):
        union |= tres_mascaras[puntuaciones.argmax()].numpy()   # 3 máscaras por caja: la mejor
    return union


def segmentar_por_texto(imagen, texto, umbral=0.35):
    """Grounding DINO busca lo que nombras y SAM 2 lo recorta."""
    entradas = dino_processor(images=imagen, text=texto, return_tensors="pt").to(device)
    with torch.no_grad():
        salidas = dino_model(**entradas)
    resultado = dino_processor.post_process_grounded_object_detection(
        salidas, entradas.input_ids, threshold=umbral, target_sizes=[(imagen.height, imagen.width)])[0]
    print("Grounding DINO encuentra:", resultado["labels"])
    mascara = mascaras_de_cajas(imagen, resultado["boxes"].tolist())
    mostrar_mascara(imagen, mascara, f'"{texto}"')
    return mascara


_ = segmentar_por_texto(image_person_cars, "yellow car. woman.")
_ = segmentar_por_texto(image_fruits, "strawberry.")
# _ = segmentar_por_texto(image_fruits3, "watermelon.")
# _ = segmentar_por_texto(image_cars, "red car.")

## Segmentar con instrucciones: Qwen3.5 + SAM 2

Grounding DINO entiende nombres, pero no instrucciones. Si cambiamos el detector por el VLM del cuaderno 3, podemos pedir cosas como "los cítricos" o "el chip negro de la placa".

Es el mismo encadenamiento: Qwen devuelve las cajas en JSON (de 0 a 1000, que convertimos a píxeles) y SAM las recorta.

Qwen3.5 son 4,5 GB de descarga: la primera carga tarda unos minutos.

In [ ]:
from transformers import AutoModelForImageTextToText

vlm_name = "Qwen/Qwen3.5-2B"
vlm_processor = AutoProcessor.from_pretrained(vlm_name)
vlm_model = AutoModelForImageTextToText.from_pretrained(vlm_name, dtype=dtype, device_map="auto")
print(f"{vlm_name} cargado")

In [ ]:
# Los "..." importan: si el ejemplo dijera "name", el modelo copiaría esa palabra como etiqueta.
FORMATO = ' Return only JSON in this format: [{"bbox_2d": [x1, y1, x2, y2], "label": "..."}]'


def preguntar(imagen, pregunta):
    """Envía una imagen y una pregunta al VLM y devuelve su respuesta."""
    mensajes = [{"role": "user", "content": [{"type": "image", "image": imagen}, {"type": "text", "text": pregunta}]}]
    entradas = vlm_processor.apply_chat_template(
        mensajes, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(vlm_model.device)
    with torch.no_grad():
        generado = vlm_model.generate(**entradas, max_new_tokens=512)
    return vlm_processor.decode(generado[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True)


def leer_json(texto):
    """Lee la lista JSON de la respuesta, venga o no entre ```json y ```."""
    texto = texto.strip().removeprefix("```json").removesuffix("```")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return []


def segmentar_con_vlm(imagen, instruccion):
    """Qwen lee la instrucción y devuelve cajas; SAM 2 las convierte en máscaras."""
    respuesta = preguntar(imagen, instruccion + FORMATO)
    print("Qwen responde:", respuesta)
    cajas = []
    for objeto in leer_json(respuesta):
        x1, y1, x2, y2 = objeto["bbox_2d"]   # de 0 a 1000: lo pasamos a píxeles
        cajas.append([x1 / 1000 * imagen.width, y1 / 1000 * imagen.height,
                      x2 / 1000 * imagen.width, y2 / 1000 * imagen.height])
    mascara = mascaras_de_cajas(imagen, cajas)
    mostrar_mascara(imagen, mascara, instruccion)
    return mascara


_ = segmentar_con_vlm(image_fruits, "Detect the citrus fruits.")
_ = segmentar_con_vlm(image_board, "Detect the black chip.")
# _ = segmentar_con_vlm(image_person_cars, "Detect the woman's bag.")
# _ = segmentar_con_vlm(image_fruits3, "Detect the watermelons.")

## Búsqueda semántica: SAM + CLIP

Otra forma de segmentar con palabras, sin detector: SAM recorta **todos** los objetos de la imagen en modo automático, y CLIP, el del cuaderno 3, dice cuál de esos recortes encaja mejor con el texto. Es la base de los buscadores que encuentran "el bolso" dentro de una foto.

Para el modo automático usamos el SAM original (`facebook/sam-vit-base`, 375 MB): en esta tarea encuentra más objetos que SAM 2 small.

In [ ]:
from transformers import CLIPModel, CLIPProcessor, pipeline

generador = pipeline("mask-generation", model="facebook/sam-vit-base", device=device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
print("SAM (modo automático) y CLIP cargados")

In [ ]:
def recortar(imagen, mascara):
    """Devuelve solo el objeto, con el fondo transparente, recortado a su tamaño."""
    filas, columnas = np.where(mascara)
    rgba = np.dstack([np.array(imagen), mascara * 255]).astype(np.uint8)   # 4.º canal: transparencia
    return Image.fromarray(rgba[filas.min():filas.max() + 1, columnas.min():columnas.max() + 1])


def buscar_objeto(imagen, texto):
    """SAM recorta todos los objetos y CLIP elige el que mejor encaja con el texto."""
    mascaras = [np.array(m, dtype=bool) for m in generador(imagen, points_per_batch=64)["masks"]]

    # Cada objeto, recortado y sobre fondo blanco, para que CLIP vea solo el objeto
    recortes = []
    for mascara in mascaras:
        recorte = recortar(imagen, mascara)
        fondo = Image.new("RGB", recorte.size, "white")
        fondo.paste(recorte, mask=recorte)
        recortes.append(fondo)

    entradas = clip_processor(text=[texto], images=recortes, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        parecido = clip_model(**entradas).logits_per_text[0]
    mejor = parecido.argmax().item()

    todas = np.zeros((imagen.height, imagen.width, 3))
    for mascara in mascaras:
        todas[mascara] = np.random.rand(3)   # un color al azar por objeto
    plt.imshow(todas)
    plt.title(f"SAM encuentra {len(mascaras)} objetos")
    plt.axis("off")
    plt.show()
    mostrar_mascara(imagen, mascaras[mejor], f'El que CLIP elige para "{texto}"')
    return mascaras[mejor]


_ = buscar_objeto(image_person_cars, "a yellow taxi")
_ = buscar_objeto(image_person_cars, "a handbag")
# _ = buscar_objeto(image_person_cars, "a red car")
# _ = buscar_objeto(image_fruits3, "a watermelon")

## Extra: el perímetro del lago de Central Park

Si vamos bien de tiempo lo hacemos juntos en clase; si no, queda para casa. Sustituye los `None` y quita el `#` del último `print`.

Con una máscara se puede medir. Vamos a calcular el perímetro del lago más grande de Central Park, el *Reservoir*, a partir de una foto de satélite.

El truco es la escala: Central Park mide **4 km** de norte a sur. Si sabemos cuántos píxeles ocupa ese largo en la foto, sabemos cuántos kilómetros mide un píxel, y con eso pasamos a kilómetros cualquier medida.

1. Segmenta el parque y el lago (ya está hecho en la primera celda).
2. Mide el largo del parque en píxeles y calcula los kilómetros por píxel.
3. Mide el perímetro del lago en píxeles (también está hecho) y pásalo a kilómetros.

Para comprobar tu resultado: el camino que rodea el Reservoir mide 1,58 millas, unos **2,5 km**.

In [ ]:
mascara_parque = segmentar_con_vlm(image_park, "Detect Central Park, the long green rectangle.")
mascara_lago = segmentar_con_vlm(image_park, "Detect the largest lake inside the park.")

In [ ]:
import cv2

LARGO_REAL_KM = None          # <- largo real de Central Park, de norte a sur

filas_parque = np.where(mascara_parque.any(axis=1))[0]   # filas de la imagen donde hay parque
largo_px = None               # <- largo del parque en píxeles (pista: última fila menos primera fila)
km_por_pixel = None           # <- kilómetros que mide un píxel

# Perímetro del lago en píxeles: el contorno de la mancha más grande de la máscara
contornos, _ = cv2.findContours(mascara_lago.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
perimetro_px = cv2.arcLength(max(contornos, key=cv2.contourArea), True)

perimetro_km = None           # <- perímetro del lago en kilómetros

print(f"Perímetro del lago: {perimetro_px:.0f} píxeles")
# print(f"Perímetro del lago: {perimetro_km:.2f} km (el real es unos 2,5 km)")

## ¿Y SAM 3?

SAM 3 (Meta, noviembre de 2025) entiende texto por sí mismo: con una sola llamada encuentra y segmenta todas las instancias de un concepto, como "watermelon", sin encadenar un detector delante. Ya está en `transformers` (`Sam3Model`), pero sus pesos tienen acceso restringido: hay que pedir permiso a Meta en la ficha del modelo y esperar a que lo aprueben. Por eso en clase usamos SAM 2 con Grounding DINO o Qwen, que dan un resultado parecido sin pedir permisos.

## Resumen

- SAM 2 segmenta cualquier objeto a partir de puntos o cajas, sin saber qué es.
- Devuelve tres máscaras por indicación, y nos quedamos con la de mejor calidad.
- Los puntos negativos quitan partes que no queremos.
- Una máscara sirve para medir: áreas, longitudes, perímetros.
- Encadenado con Grounding DINO segmenta lo que nombras; con Qwen, lo que describes.
- En modo automático SAM recorta todos los objetos, y con CLIP se busca entre ellos el que encaja con un texto.

**Siguiente:** cuaderno 6, Otras tareas de visión. Antes, el tema 6 de la web. [Abrir el cuaderno en Colab](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/6_Otras_Tareas.ipynb).